# Model 1 — AEF → Forest Classifier (PyTorch + GPU)
Trains an MLP on 64D AEF embeddings with WorldCover 2020 as ground truth.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import rasterio
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

from config import S2_TRAIN, WORLDCOVER_DIR, MODEL1_DIR, FUSED_LABELS_DIR
from loader import load_aef

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Discover tiles

In [ ]:
WC_YEAR = 2020
AEF_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
s2_root = pathlib.Path(S2_TRAIN)
wc_root = pathlib.Path(WORLDCOVER_DIR)

tile_ids = sorted([
    d.name.replace('__s2_l2a', '')
    for d in s2_root.iterdir()
    if d.is_dir() and d.name.endswith('__s2_l2a')
])
tiles_with_wc = [t for t in tile_ids
                 if (wc_root / f'{t}_worldcover_{WC_YEAR}.tif').exists()]
print(f'Tiles with WorldCover: {len(tiles_with_wc)}/{len(tile_ids)}')

## 2. Load data (100k samples/tile to avoid OOM)

In [ ]:
def get_s2_meta(tile_id):
    tif = sorted((s2_root / f'{tile_id}__s2_l2a').glob('*.tif'))[0]
    with rasterio.open(tif) as src: return src.meta.copy()

def load_worldcover(tile_id):
    with rasterio.open(wc_root / f'{tile_id}_worldcover_{WC_YEAR}.tif') as src:
        return src.read(1)

SAMPLES_PER_TILE = 100_000
rng = np.random.default_rng(42)
X_list, y_list, tile_index = [], [], []

for tile_id in tiles_with_wc:
    print(f'Loading {tile_id} ...', end=' ')
    ref_meta = get_s2_meta(tile_id)
    aef = load_aef(tile_id, WC_YEAR, ref_meta, 'train')
    wc  = load_worldcover(tile_id)
    if aef is None: print('SKIP'); continue
    X = aef.reshape(64, -1).T
    y = wc.reshape(-1).astype(np.float32)
    n = min(SAMPLES_PER_TILE, len(y))
    idx = rng.choice(len(y), size=n, replace=False)
    X_list.append(X[idx]); y_list.append(y[idx])
    tile_index.extend([tile_id] * n)
    print(f'{y.mean()*100:.1f}% forest')
    del aef, X, y

X_all = np.concatenate(X_list)
y_all = np.concatenate(y_list)
tile_index = np.array(tile_index)
print(f'Total: {len(y_all):,}  |  Forest: {y_all.mean()*100:.1f}%')

## 3. Spatial train / val split

In [ ]:
VAL_TILES = [t for t in ['18NXJ_7_6', '47QQV_2_4', '48PYB_3_6'] if t in tiles_with_wc]
is_val = np.isin(tile_index, VAL_TILES)
X_train, y_train = X_all[~is_val], y_all[~is_val]
X_val,   y_val   = X_all[is_val],  y_all[is_val]
print(f'Train: {len(y_train):,}  Val: {len(y_val):,}')

## 4. Normalize

In [ ]:
# Fill NaN from reprojection borders, then normalize and clip
X_train = np.nan_to_num(X_train, nan=0.0)
X_val   = np.nan_to_num(X_val,   nan=0.0)

mean = X_train.mean(axis=0)
std  = X_train.std(axis=0) + 1e-8
X_train_n = np.clip((X_train - mean) / std, -10, 10).astype(np.float32)
X_val_n   = np.clip((X_val   - mean) / std, -10, 10).astype(np.float32)

assert not np.isnan(X_train_n).any(), "NaN still present after nan_to_num"
print(f"X range: [{X_train_n.min():.2f}, {X_train_n.max():.2f}]")

import pathlib
pathlib.Path(MODEL1_DIR).mkdir(parents=True, exist_ok=True)
np.save(f"{MODEL1_DIR}/feature_mean.npy", mean)
np.save(f"{MODEL1_DIR}/feature_std.npy",  std)
print("Normalisation stats saved.")

## 5. Train MLP on GPU

In [ ]:
class ForestMLP(nn.Module):
    def __init__(self, in_dim=64, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(1)

def make_loader(X, y, batch_size=65536, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))

model = ForestMLP().to(device)
# No pos_weight: forest is majority class in tropical tiles — standard BCE is correct here
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
EPOCHS = 20
train_loader = make_loader(X_train_n, y_train)
val_loader   = make_loader(X_val_n,   y_val, shuffle=False)
train_losses, val_f1s = [], []

print(f"Training on: {device}  ({len(y_train):,} samples)")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
    scheduler.step()
    train_losses.append(total_loss / len(y_train))

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            preds.append((model(Xb.to(device)).sigmoid() > 0.5).cpu().numpy())
            trues.append(yb.numpy())
    f1 = f1_score(np.concatenate(trues), np.concatenate(preds), zero_division=0)
    val_f1s.append(f1)
    print(f"Epoch {epoch:2d}/{EPOCHS}  loss={train_losses[-1]:.4f}  val_F1={f1:.4f}")

print("Done.")

In [ ]:
print(classification_report(np.concatenate(trues), np.concatenate(preds),
                             target_names=['non-forest', 'forest']))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax2.plot(val_f1s);      ax2.set_title('Val F1'); ax2.set_xlabel('Epoch')
plt.tight_layout()
pathlib.Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/model1_training.png', dpi=100); plt.show()

## 6. Save model

In [ ]:
model_path = pathlib.Path(MODEL1_DIR) / 'forest_mlp.pt'
torch.save(model.state_dict(), model_path)
print(f'Saved -> {model_path}')

## 7. Visualise predictions on a validation tile

In [ ]:
VIZ_TILE = VAL_TILES[0] if VAL_TILES else tiles_with_wc[0]
ref_meta = get_s2_meta(VIZ_TILE)
H, W = ref_meta['height'], ref_meta['width']
aef_2020 = load_aef(VIZ_TILE, 2020, ref_meta, 'train')
wc_gt    = load_worldcover(VIZ_TILE)

X_viz = ((aef_2020.reshape(64, -1).T - mean) / std).astype(np.float32)
model.eval()
with torch.no_grad():
    probs = model(torch.from_numpy(X_viz).to(device)).sigmoid().cpu().numpy()
prob_map = probs.reshape(H, W)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(wc_gt,          cmap='Greens', vmin=0, vmax=1); axes[0].set_title(f'WorldCover GT ({VIZ_TILE})')
axes[1].imshow(prob_map,       cmap='Greens', vmin=0, vmax=1); axes[1].set_title('Predicted probability')
axes[2].imshow(prob_map > 0.5, cmap='Greens', vmin=0, vmax=1); axes[2].set_title('Predicted mask (>0.5)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.savefig('outputs/model1_viz.png', dpi=100); plt.show()

## 8. Apply to all years → yearly forest probability maps

In [ ]:
prob_dir = pathlib.Path(MODEL1_DIR) / 'forest_probs'
prob_dir.mkdir(parents=True, exist_ok=True)
model.eval()

for tile_id in tqdm(tile_ids, desc='Tiles'):
    ref_meta = get_s2_meta(tile_id)
    H, W = ref_meta['height'], ref_meta['width']
    for year in AEF_YEARS:
        out_path = prob_dir / f'{tile_id}_forest_prob_{year}.tif'
        if out_path.exists(): continue
        aef = load_aef(tile_id, year, ref_meta, 'train')
        if aef is None: continue
        X = ((aef.reshape(64, -1).T - mean) / std).astype(np.float32)
        chunks = []
        with torch.no_grad():
            for i in range(0, len(X), 65536):
                chunks.append(model(torch.from_numpy(X[i:i+65536]).to(device)).sigmoid().cpu().numpy())
        probs = np.concatenate(chunks).reshape(H, W).astype(np.float32)
        del aef, X
        out_meta = ref_meta.copy(); out_meta.update(count=1, dtype='float32', nodata=-1)
        with rasterio.open(out_path, 'w', **out_meta) as dst: dst.write(probs, 1)

print('Done ->', prob_dir)

## 9. Derive improved deforestation labels
Forest in 2020 AND lost forest in any year 2021–2024.

In [ ]:
FOREST_THRESHOLD = 0.5
DEFOR_YEARS = [2021, 2022, 2023, 2024]
improved_dir = pathlib.Path(MODEL1_DIR) / 'improved_labels'
improved_dir.mkdir(parents=True, exist_ok=True)

for tile_id in tqdm(tile_ids, desc='Tiles'):
    p2020 = prob_dir / f'{tile_id}_forest_prob_2020.tif'
    if not p2020.exists(): continue
    with rasterio.open(p2020) as src:
        was_forest = src.read(1) >= FOREST_THRESHOLD
        ref_meta = src.meta.copy()
    lost = np.zeros(was_forest.shape, bool)
    for yr in DEFOR_YEARS:
        p = prob_dir / f'{tile_id}_forest_prob_{yr}.tif'
        if not p.exists(): continue
        with rasterio.open(p) as src: lost |= (src.read(1) < FOREST_THRESHOLD)
    label = (was_forest & lost).astype(np.uint8)
    ref_meta.update(count=1, dtype='uint8', nodata=None)
    with rasterio.open(improved_dir / f'{tile_id}_improved_label.tif', 'w', **ref_meta) as dst:
        dst.write(label, 1)
    print(f'{tile_id}: {label.mean()*100:.2f}% deforestation')
print('Improved labels ->', improved_dir)

## 10. Compare improved vs majority-vote labels

In [ ]:
import random
compare_tile  = random.choice(tile_ids)
fused_path    = pathlib.Path(FUSED_LABELS_DIR) / f'{compare_tile}_fused.tif'
improved_path = improved_dir / f'{compare_tile}_improved_label.tif'
if fused_path.exists() and improved_path.exists():
    with rasterio.open(fused_path)    as src: fused    = src.read(1)
    with rasterio.open(improved_path) as src: improved = src.read(1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(fused,    cmap='hot', vmin=0, vmax=1)
    axes[0].set_title(f'Majority-vote  {compare_tile}  ({fused.mean()*100:.2f}%)')
    axes[1].imshow(improved, cmap='hot', vmin=0, vmax=1)
    axes[1].set_title(f'Model 1 improved  ({improved.mean()*100:.2f}%)')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.savefig('outputs/model1_comparison.png', dpi=100); plt.show()
else:
    print('Run fuse_labels.py first for the comparison.')